In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv("../../dataset/Cleaned_Suicide_Detection_DL.csv")
df.columns

Index(['text', 'class', 'clean_text', 'labels'], dtype='object')

In [3]:
X = df["clean_text"]
y = df["labels"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

In [ ]:
model = Sequential([
    Embedding(20000, 128),
    Bidirectional(LSTM(128)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.fit(X_train_pad, y_train, epochs=5, batch_size=32, validation_split=0.1)

Epoch 1/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 717s 136ms/step - accuracy: 0.9132 - loss: 0.2334 - val_accuracy: 0.9323 - val_loss: 0.1812
Epoch 2/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 710s 136ms/step - accuracy: 0.9383 - loss: 0.1674 - val_accuracy: 0.9359 - val_loss: 0.1721
Epoch 3/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 784s 150ms/step - accuracy: 0.9474 - loss: 0.1407 - val_accuracy: 0.9359 - val_loss: 0.1713
Epoch 4/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 766s 147ms/step - accuracy: 0.9550 - loss: 0.1185 - val_accuracy: 0.9357 - val_loss: 0.1830
Epoch 5/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 740s 142ms/step - accuracy: 0.9622 - loss: 0.0992 - val_accuracy: 0.9343 - val_loss: 0.2075


In [14]:
loss, acc = model.evaluate(X_test_pad, y_test)
print(acc)

1451/1451 ━━━━━━━━━━━━━━━━━━━━ 53s 37ms/step - accuracy: 0.9311 - loss: 0.2172
0.9310590028762817


In [16]:
model.save("../../models/bilstm_model.h5")

In [6]:
model = load_model("../../models/bilstm_model.h5")

sample = ["I want to die"]
sample = tokenizer.texts_to_sequences(sample)
sample = pad_sequences(sample, maxlen=100)

pred = model.predict(sample)[0][0]
pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step


np.float32(0.92679185)

In [7]:
y_pred = model.predict(X_test_pad)
y_pred = (y_pred > 0.5).astype(int)
y_pred

1451/1451 ━━━━━━━━━━━━━━━━━━━━ 34s 23ms/step


array([[1],
       [0],
       [1],
       ...,
       [1],
       [1],
       [0]], shape=(46402, 1))

In [8]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93     23257
           1       0.92      0.94      0.93     23145

    accuracy                           0.93     46402
   macro avg       0.93      0.93      0.93     46402
weighted avg       0.93      0.93      0.93     46402

